In [1]:
import json
from pathlib import Path

import numpy as np

from pyscf import gto

import sys

BASIS = "cc-pVTZ"


def gen_basis(basis):
    return {
        "H": basis,
        "He": basis,
        "Li": basis,
        "Be": basis,
        "B": basis,
        "C": basis,
        "N": basis,
        "O": basis,
        "F": basis,
        "Ne": basis,
        "Na": basis,
        "Mg": basis,
        "Al": basis,
        "Si": basis,
        "P": basis,
        "S": basis,
        "Cl": basis,
        "Ar": basis,
        # "K": basis,
        "Ca": basis,
        "Sc": basis,
        "Ti": basis,
        "V": basis,
        "Cr": basis,
        "Mn": basis,
        "Fe": basis,
        "Co": basis,
        "Ni": basis,
        "Cu": basis,
        "Zn": basis,
        "Ga": basis,
        "Ge": basis,
        "As": basis,
        "Se": basis,
        "Br": basis,
        "Kr": basis,
        "Y": f"{basis}-PP",
        "Zr": f"{basis}-PP",
        "Nb": f"{basis}-PP",
        "Mo": f"{basis}-PP",
        "Tc": f"{basis}-PP",
        "Ru": f"{basis}-PP",
        "Rh": f"{basis}-PP",
        "Pd": f"{basis}-PP",
        "Ag": f"{basis}-PP",
        "Cd": f"{basis}-PP",
        "In": f"{basis}-PP",
        "Sn": f"{basis}-PP",
        "Sb": f"{basis}-PP",
        "Te": f"{basis}-PP",
        "I": f"{basis}-PP",
        "Xe": f"{basis}-PP",
        "Hf": f"{basis}-PP",
        "Ta": f"{basis}-PP",
        "W": f"{basis}-PP",
        "Re": f"{basis}-PP",
        "Os": f"{basis}-PP",
        "Ir": f"{basis}-PP",
        "Pt": f"{basis}-PP",
        "Au": f"{basis}-PP",
        "Hg": f"{basis}-PP",
        "Tl": f"{basis}-PP",
        "Pb": f"{basis}-PP",
        "Bi": f"{basis}-PP",
        "Po": f"{basis}-PP",
        "At": f"{basis}-PP",
        "Rn": f"{basis}-PP",
    }


def gen_ecp(basis):
    return {
        "Y": f"{basis}-PP",
        "Zr": f"{basis}-PP",
        "Nb": f"{basis}-PP",
        "Mo": f"{basis}-PP",
        "Tc": f"{basis}-PP",
        "Ru": f"{basis}-PP",
        "Rh": f"{basis}-PP",
        "Pd": f"{basis}-PP",
        "Ag": f"{basis}-PP",
        "Cd": f"{basis}-PP",
        "In": f"{basis}-PP",
        "Sn": f"{basis}-PP",
        "Sb": f"{basis}-PP",
        "Te": f"{basis}-PP",
        "I": f"{basis}-PP",
        "Xe": f"{basis}-PP",
        "Hf": f"{basis}-PP",
        "Ta": f"{basis}-PP",
        "W": f"{basis}-PP",
        "Re": f"{basis}-PP",
        "Os": f"{basis}-PP",
        "Ir": f"{basis}-PP",
        "Pt": f"{basis}-PP",
        "Au": f"{basis}-PP",
        "Hg": f"{basis}-PP",
        "Tl": f"{basis}-PP",
        "Pb": f"{basis}-PP",
        "Bi": f"{basis}-PP",
        "Po": f"{basis}-PP",
        "At": f"{basis}-PP",
        "Rn": f"{basis}-PP",
    }

In [2]:
dict_ = {"molecule": [], "spin": {}, "charge": {}}

data_path = Path("./sets")
# download the data from https://github.com/obackhouse/gmtkn
# Note
#   1) The data is not included in this repository.
#   2) The data included GMTKN55, GW100, GAPS, MRADC, ACC24, S30L, BH9.
max_num_atom = -1

for python_path in data_path.glob("*.py"):
    if python_path.name == "__init__.py":
        continue
    exec(python_path.read_text())
    # load the systems and reactions from the executed python file
    dataset = python_path.stem
    print(f"Processing {dataset}")
    dict_[f"molecule_{dataset}"] = []

    for i_name in systems:
        molecule = []
        for i_atom in range(len(systems[i_name]["atoms"])):
            molecule.append(
                [
                    systems[i_name]["atoms"][i_atom],
                    float(systems[i_name]["coords"][i_atom][0]),
                    float(systems[i_name]["coords"][i_atom][1]),
                    float(systems[i_name]["coords"][i_atom][2]),
                ]
            )

        try:
            mol = gto.M(
                atom=molecule,
                basis=gen_basis(BASIS),
                ecp=gen_ecp(BASIS),
                verbose=0,
                charge=systems[i_name]["charge"],
                spin=systems[i_name]["spin"],
                unit="B",
            )
            mol.build()
            max_num_atom = max(max_num_atom, len(molecule))
        except Exception as e:
            print(molecule)
            print(f"Error in {i_name}: {e}")
            continue

        molecule = []
        for i_atom in mol._atom:
            molecule.append(
                [
                    i_atom[0],
                    i_atom[1][0],
                    i_atom[1][1],
                    i_atom[1][2],
                ]
            )

        dict_i_name = f"{dataset}-{i_name}"
        dict_[dict_i_name] = molecule
        dict_[f"molecule_{dataset}"].append(dict_i_name)
        dict_["molecule"].append(dict_i_name)
        dict_["charge"][dict_i_name] = systems[i_name]["charge"]
        dict_["spin"][dict_i_name] = systems[i_name]["spin"]

    dict_[f"reaction-{dataset}"] = {}
    for i_reaction, reaction in enumerate(reactions):
        dict_[f"reaction-{dataset}"][i_reaction] = {
            "systems": reaction["systems"],
            "stoichiometry": reaction["stoichiometry"],
            "reference": reaction["reference"],
        }

    for number_atom in range(max_num_atom + 1):
        dict_[f"molecule{number_atom}-{dataset}"] = []

    for molecule in dict_[f"molecule_{dataset}"]:
        num_heavy_atom = 0
        num_atom = 0
        for atom in dict_[molecule]:
            num_atom += 1
            if atom[0].upper() != "H":
                num_heavy_atom += 1

        if num_heavy_atom == 0:
            if num_atom == 1:
                dict_[f"molecule0-{dataset}"].append(molecule)
            else:
                dict_[f"molecule1-{dataset}"].append(molecule)
        elif num_heavy_atom == 1:
            if num_atom == 1:
                dict_[f"molecule0-{dataset}"].append(molecule)
            else:
                dict_[f"molecule1-{dataset}"].append(molecule)
        else:
            dict_[f"molecule{num_heavy_atom}-{dataset}"].append(molecule)

for number_atom in range(max_num_atom):
    dict_[f"molecule{number_atom}"] = []

for molecule in dict_["molecule"]:
    num_heavy_atom = 0
    num_atom = 0
    for atom in dict_[molecule]:
        num_atom += 1
        if atom[0].upper() != "H":
            num_heavy_atom += 1

    if num_heavy_atom == 0:
        if num_atom == 1:
            dict_[f"molecule0"].append(molecule)
        else:
            dict_[f"molecule1"].append(molecule)
    elif num_heavy_atom == 1:
        if num_atom == 1:
            dict_[f"molecule0"].append(molecule)
        else:
            dict_[f"molecule1"].append(molecule)
    else:
        dict_[f"molecule{num_heavy_atom}"].append(molecule)
print()
print("DONE")

Processing BUT14DIOL
Processing C60ISO
Processing SCONF
Processing RG18
Processing TAUT15
Processing BSR36
Processing PNICO23
Processing ADIM6
Processing MCONF
Processing ISO34
Processing NBPRC
Processing YBDE18
Processing G2RC
Processing DIPCS10
Processing WATER27
Processing CDIE20
Processing WCPT18
Processing DARC
Processing S22
Processing HAL59
Processing ICONF
Processing HEAVY28
Processing DC13
Processing HEAVYSB11
Processing BHROT27
Processing INV24
Processing SIE4x4
Processing ISOL24
Processing IDISP
Processing CHB6


Processing FH51
Processing BHDIV10
Processing G21IP
Processing PX13
Processing ACONF
Processing AL2X6
Processing ALKBDE10


Processing UPU23
Processing MB16_43
Processing PA26
Processing BHPERI
Processing RSE43
Processing PCONF21
Processing CARBHB12
Processing RC21
Processing W4_11
Processing IL16
Processing S66
Processing G21EA
Processing Amino20x4
Processing ALK8
Processing PArel
Processing BH76
Processing AHB21

DONE


In [3]:
dict_new = {}
for key in dict_:
    if key.startswith("molecule"):
        if dict_[key]:
            dict_new[key] = dict_[key]
    else:
        dict_new[key] = dict_[key]

with open(f"gmtkn-{BASIS}.json", "w") as f:
    json.dump(dict_new, f)

In [ ]:
# # for clean up
# import json
# import re
# import sys

# import numpy as np


# sys.path.append("../cadft")
# from cc2cc.utils.basis import gen_basis
# from cc2cc.utils.rotate import rotate

# dict_new = {"molecule": [], "spin": {}, "charge": {}}

# with open(f"gmtkn-{BASIS}.json", "r") as f:
#     dict_ = json.load(f)

# for key in dict_:
#     if re.match(r"molecule[0-9]+", key):
#         dict_new[key] = []
#         print(f"Processing {dict_[key]}")
#         for molecule in dict_[key]:
#             molecule_list = np.array(dict_[molecule], dtype=object)
#             molecule_list = molecule_list[np.argsort(molecule_list[:, 0])]
#             rotate(molecule_list)

#             add_flag = True

#             duplicate_flag = False
#             for molecule_new in dict_new[key]:
#                 molecule_new_list = np.array(dict_[molecule_new], dtype=object)
#                 molecule_new_list = molecule_new_list[
#                     np.argsort(molecule_new_list[:, 0])
#                 ]
#                 rotate(molecule_new_list)

#                 if len(molecule_new_list) != len(molecule_list):
#                     continue

#                 for i_atom in range(len(molecule_list)):
#                     if (
#                         molecule_new_list[i_atom][0] == molecule_list[i_atom][0]
#                         and (
#                             dict_["charge"][molecule] == dict_["charge"][molecule_new]
#                             and dict_["spin"][molecule] == dict_["spin"][molecule_new]
#                         )
#                         and np.linalg.norm(
#                             molecule_new_list[i_atom][1:] - molecule_list[i_atom][1:]
#                         )
#                         < 1e-7
#                     ):
#                         duplicate_flag = True
#                     else:
#                         duplicate_flag = False
#                         break

#                 if duplicate_flag:
#                     add_flag = False
#                     break

#             if add_flag:
#                 dict_new[key].append(molecule)
#                 dict_new[molecule] = dict_[molecule]
#                 dict_new["molecule"].append(molecule)
#                 dict_new["charge"][molecule] = dict_["charge"][molecule]
#                 dict_new["spin"][molecule] = dict_["spin"][molecule]
#             else:
#                 print(
#                     f"spin and charge: [{dict_['spin'][molecule]}, {dict_['charge'][molecule]}] with [{dict_['spin'][molecule_new]}, {dict_['charge'][molecule_new]}]"
#                 )
#                 print(
#                     f"Comparing atom and coordinates: \n {molecule_new_list} \n and \n {molecule_list}"
#                 )
#                 print(f"Duplicate found: {molecule} and {molecule_new}\n")
#                 dict_new[molecule] = molecule_new

#         print(len(dict_[key]), len(dict_new[key]))
#         print(f"remove duplicates: {[_ for _ in dict_[key] if _ not in dict_new[key]]}")
#         print()
#     elif re.match(r"reaction-.*", key):
#         dict_new[key] = dict_[key]
#         print(f"Reaction '{key}'.")
#     elif re.match(r"molecule_.*", key):
#         dict_new[key] = dict_[key]
#         print(f"molecule key '{key}' processed.")

# with open(f"filtered_gmtkn-{BASIS}.json", "w") as f:
#     json.dump(dict_new, f)

# print(f"Filtered data has been saved to filtered_gmtkn-{BASIS}.json")
# print("DONE")

ModuleNotFoundError: No module named 'cc2cc.utils.basis'